# Day 4 | Lab 4.3: Complete LCEL RAG Pipeline — Banking Policy Q&A

**Duration:** ~1.5 hours

**Scenario.** A retail-banking knowledge-base assistant — trainees ask natural-language questions ("what's the prepayment fee on a personal loan?") and the system retrieves the relevant policy chunks, grounds the answer in them, and returns the answer **with source citations**. Banking framing preserved verbatim from the AGN source notebook.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Build an end-to-end RAG pipeline with **LCEL** (LangChain Expression Language) — `retriever | prompt | llm | parser`.
2. Use **FAISS** (in-memory) and **ChromaDB** (persistent) as interchangeable vector stores behind a `Retriever`.
3. Pick a sensible chunk size + retrieval `k` for a banking-policy corpus.
4. Return **source citations** in the final answer — page number, document ID, section.
5. Save and reload FAISS / ChromaDB indexes so production serving doesn't re-embed every cold start.
6. Recognise where conversational RAG fits — and use the modern memory pattern from Lab 3.1 (no deprecated `ConversationalRetrievalChain`).

**Tools.** LangChain v1 · FAISS · ChromaDB · OpenAI (`text-embedding-3-small` + `gpt-4.1-mini`).

*Adapted from AGN RAG System: `(NEW) RAG-Hands-On-1-FAISS-ChromaDB-OpenAI-LCEL.ipynb`. Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 📦 Step 1: Installation & Setup

Install all required packages:

In [ ]:
import langchain
langchain.__version__

'1.2.10'

### Import Libraries

In [ ]:
import os
from dotenv import load_dotenv
from typing import List, Dict
import time

# LangChain / OpenAI imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS, Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


### Load Environment Variables

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


---
## 2. Sample Banking Policy Corpus

---

## 📄 Step 2: Create Sample Financial Documents

We'll create realistic banking policy documents covering:
- Commercial loan requirements
- Mortgage lending guidelines
- Credit card policies
- Business account terms
- Investment services

In [ ]:
# Sample banking policy documents
financial_documents = [
    {
        "title": "Commercial Loan Policy 2024",
        "content": """
        COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum 2 years
        - Annual revenue: Minimum $500,000

        Loan Amounts:
        - Minimum: $50,000
        - Maximum: $5,000,000
        - Term length: 5-25 years

        Interest Rates (as of November 2024):
        - Prime Rate + 2.5% to 5.5%
        - Current range: 8.5% - 11.5% APR
        - Fixed or variable rate options available

        Required Documentation:
        - Last 3 years business tax returns
        - Current year-to-date profit & loss statement
        - Business bank statements (6 months)
        - Business plan and financial projections
        - Personal financial statement from all owners with >20% stake

        Collateral Requirements:
        - Real estate: 75-80% loan-to-value ratio
        - Equipment: 70-75% loan-to-value ratio
        - Inventory: 50-60% loan-to-value ratio
        - Personal guarantee required for loans >$250,000

        Processing Timeline:
        - Application to approval: 15-30 business days
        - Closing: 5-10 business days after approval

        Fees:
        - Origination fee: 1-2% of loan amount
        - Application fee: $500 (non-refundable)
        - Appraisal fee: Varies by collateral value
        """
    },
    {
        "title": "Mortgage Lending Guidelines 2024",
        "content": """
        RESIDENTIAL MORTGAGE LENDING POLICY

        Conventional Mortgages:
        - Minimum credit score: 620
        - Recommended credit score: 740+ for best rates
        - Down payment: 20% (or PMI required)
        - Debt-to-income ratio: Maximum 43%
        - Maximum loan amount: $726,200 (2024 conforming limit)

        Current Mortgage Rates (November 2024):
        - 30-year fixed: 6.5% APR
        - 15-year fixed: 5.875% APR
        - 5/1 ARM: 5.75% APR (initial rate)

        FHA Loans:
        - Minimum credit score: 580 (3.5% down) or 500 (10% down)
        - Down payment: As low as 3.5%
        - Maximum loan amount: $498,257 (most areas)
        - Mortgage insurance required (upfront + annual)
        - Current rate: 6.25% APR (30-year fixed)

        VA Loans:
        - No minimum credit score (typically 620 recommended)
        - No down payment required
        - No mortgage insurance
        - Funding fee: 2.3% (can be financed)
        - Current rate: 6.0% APR (30-year fixed)
        - Available to eligible veterans, active duty, and surviving spouses

        Jumbo Mortgages:
        - Loan amounts exceeding $726,200
        - Minimum credit score: 700
        - Down payment: Minimum 20%
        - Debt-to-income ratio: Maximum 38%
        - Current rate: 6.75% APR (30-year fixed)

        Required Documentation:
        - Last 2 years W-2s or tax returns
        - Last 2 months pay stubs
        - Last 2 months bank statements
        - Employment verification
        - Photo ID
        - List of assets and liabilities

        Closing Costs:
        - Typical range: 2-5% of loan amount
        - Includes: Origination fee, appraisal, title insurance, recording fees
        - Lender credits available for higher interest rates

        Pre-approval Process:
        - Submit application and documentation
        - Credit check and verification
        - Pre-approval letter issued within 3-5 business days
        - Valid for 90 days
        """
    },
    {
        "title": "Credit Card Policy 2024",
        "content": """
        CREDIT CARD PRODUCTS AND REQUIREMENTS

        Premium Rewards Card:
        - Annual fee: $95
        - APR: 18.99% - 25.99% (based on creditworthiness)
        - Minimum credit score: 720
        - Credit limit: $10,000 - $50,000
        - Rewards: 3% cash back on travel, 2% on dining, 1% on all other purchases
        - Sign-up bonus: $500 after spending $3,000 in first 3 months
        - Benefits: Travel insurance, purchase protection, extended warranty

        Everyday Cash Back Card:
        - Annual fee: $0
        - APR: 16.99% - 23.99%
        - Minimum credit score: 670
        - Credit limit: $2,000 - $25,000
        - Rewards: 2% cash back on groceries and gas, 1% on other purchases
        - Sign-up bonus: $200 after spending $1,000 in first 3 months

        Business Credit Card:
        - Annual fee: $125
        - APR: 17.99% - 24.99%
        - Minimum credit score: 680 (personal) and 2 years in business
        - Credit limit: $5,000 - $100,000
        - Rewards: 3% on business expenses, 2% on office supplies, 1% on other
        - Employee cards: Free (up to 10 cards)
        - Expense management tools included

        Secured Credit Card:
        - Annual fee: $0
        - APR: 21.99%
        - No minimum credit score
        - Security deposit: $200 - $2,500 (equals credit limit)
        - Rewards: 1% cash back on all purchases
        - Graduate to unsecured card after 12 months of on-time payments

        General Terms:
        - Grace period: 25 days from statement closing
        - Late payment fee: Up to $40
        - Foreign transaction fee: 0% (Premium and Business), 3% (others)
        - Balance transfer fee: 3% of amount transferred
        - Cash advance fee: 5% or $10, whichever is greater
        - Over-limit fee: $0 (we decline over-limit transactions)

        Application Process:
        - Online application: Instant decision in most cases
        - Credit check required (hard inquiry)
        - Card arrival: 7-10 business days
        """
    },
    {
        "title": "Business Banking Account Terms 2024",
        "content": """
        BUSINESS CHECKING AND SAVINGS ACCOUNTS

        Business Checking Account:
        - Monthly maintenance fee: $15 (waived with $5,000 minimum balance)
        - Transaction limit: 200 free transactions/month
        - Additional transaction fee: $0.50 per transaction over limit
        - Interest rate: 0.50% APY on balances over $10,000
        - Mobile deposit: Available
        - Online bill pay: Free
        - Debit card: Included

        Premium Business Checking:
        - Monthly maintenance fee: $30 (waived with $25,000 minimum balance)
        - Transaction limit: Unlimited
        - Interest rate: 1.25% APY on all balances
        - Cash management services: Included
        - Wire transfers: 5 free domestic per month
        - Dedicated business banker: Yes

        Business Savings Account:
        - Monthly maintenance fee: $5 (waived with $2,500 minimum balance)
        - Interest rate tiers:
          * $0 - $10,000: 2.50% APY
          * $10,001 - $50,000: 3.00% APY
          * $50,001+: 3.50% APY
        - Withdrawal limit: 6 per statement cycle
        - Excess withdrawal fee: $10 per transaction

        Business Money Market Account:
        - Monthly maintenance fee: $10 (waived with $10,000 minimum balance)
        - Minimum opening deposit: $10,000
        - Interest rate tiers:
          * $10,000 - $25,000: 3.25% APY
          * $25,001 - $100,000: 3.75% APY
          * $100,001+: 4.00% APY
        - Check writing: Limited to 10 checks per month
        - Debit card: Available

        Merchant Services:
        - Credit card processing: 2.6% + $0.10 per transaction
        - ACH processing: $0.25 per transaction
        - Monthly equipment rental: $25
        - PCI compliance support: Included
        - Same-day funding available

        Additional Services:
        - Remote deposit capture: $25/month
        - Positive pay (fraud protection): $30/month
        - Account analysis: $15/month
        - Stop payment: $25 per item
        - Returned item/NSF: $35 per item
        - Wire transfer (domestic): $25 outgoing, $15 incoming
        - Wire transfer (international): $50 outgoing, $20 incoming
        """
    },
    {
        "title": "Investment Services Guide 2024",
        "content": """
        WEALTH MANAGEMENT AND INVESTMENT SERVICES

        Brokerage Accounts:
        - Account minimum: $0 for self-directed, $25,000 for managed
        - Stock/ETF trades: $0 commission
        - Options trades: $0.65 per contract
        - Mutual funds: No transaction fee on 4,000+ funds
        - Advisory fee: 0.75% - 1.25% annually (managed accounts)

        Retirement Accounts (IRA):
        - Traditional IRA contribution limit: $7,000 (2024), $8,000 if 50+
        - Roth IRA contribution limit: Same, subject to income limits
        - Income limit for Roth (single): $153,000 - $168,000 phase-out
        - Income limit for Roth (married): $228,000 - $243,000 phase-out
        - SEP IRA: Up to 25% of compensation or $69,000
        - SIMPLE IRA: $16,000 ($19,500 if 50+) employee contribution

        Certificates of Deposit (CDs):
        - 3-month CD: 4.50% APY (minimum $1,000)
        - 6-month CD: 4.75% APY (minimum $1,000)
        - 1-year CD: 5.00% APY (minimum $1,000)
        - 2-year CD: 4.85% APY (minimum $1,000)
        - 5-year CD: 4.60% APY (minimum $1,000)
        - Jumbo CD (>$100,000): Add 0.10% to above rates
        - Early withdrawal penalty: 3-12 months interest depending on term

        Managed Portfolio Services:
        - Conservative portfolio: 70% bonds, 30% stocks (target 5-6% return)
        - Balanced portfolio: 50% bonds, 50% stocks (target 6-8% return)
        - Growth portfolio: 30% bonds, 70% stocks (target 8-10% return)
        - Aggressive portfolio: 10% bonds, 90% stocks (target 10-12% return)
        - Rebalancing: Quarterly
        - Tax-loss harvesting: Included for accounts over $100,000

        Financial Planning Services:
        - Comprehensive financial plan: $2,500 (one-time)
        - Retirement planning: $1,500 (one-time)
        - Estate planning: $2,000 (one-time)
        - Tax planning: $500 annually
        - Ongoing advisory: 1.00% of assets under management
        - Free consultation for clients with $250,000+ invested

        Education Savings (529 Plans):
        - State tax deduction available (check your state)
        - Management fee: 0.30% - 0.60% annually
        - Age-based portfolios available
        - Static portfolios available
        - Tax-free growth for qualified education expenses
        """
    },
    {
        "title": "Risk Management and Fraud Prevention 2024",
        "content": """
        FRAUD PROTECTION AND SECURITY MEASURES

        Account Security Features:
        - Multi-factor authentication (MFA): Required for all online banking
        - Biometric login: Fingerprint and face recognition available
        - Transaction alerts: Real-time via email/SMS
        - Card controls: Set spending limits, block categories, freeze card instantly
        - Travel notifications: Enable before traveling to prevent blocks

        Fraud Detection:
        - 24/7 monitoring of all transactions
        - Machine learning algorithms detect unusual patterns
        - Automatic fraud alerts sent immediately
        - Zero liability for unauthorized transactions reported within 60 days
        - Provisional credit issued within 10 business days of report

        Identity Theft Protection:
        - Free credit monitoring (for premium account holders)
        - Dark web monitoring included
        - Identity theft insurance: Up to $1 million coverage
        - Resolution specialists available
        - Annual fee: $15/month (free for accounts with $50,000+ balance)

        Business Fraud Prevention:
        - Positive Pay: Match checks against issued check register
        - ACH Positive Pay: Approve ACH debits before processing
        - Dual authorization: Require two approvers for large transactions
        - IP address restrictions: Limit access to specific locations
        - Role-based access: Control who can do what

        Cybersecurity Best Practices:
        - Use strong, unique passwords (minimum 12 characters)
        - Enable MFA on all accounts
        - Never share passwords or security codes
        - Beware of phishing emails and calls
        - Keep devices and software updated
        - Use secure, private Wi-Fi for banking
        - Review statements monthly for unauthorized activity

        Reporting Fraud:
        - Lost/stolen card: Call 1-800-BANK-911 immediately (24/7)
        - Suspicious transactions: Report within 60 days for full protection
        - Online fraud: Email security@bank.com or call fraud hotline
        - Business account compromise: Call dedicated business line
        - Response time: Immediate card deactivation, investigation within 10 days
        """
    }
]

print(f"✅ Created {len(financial_documents)} banking policy documents")
print("\nDocuments:")
for i, doc in enumerate(financial_documents, 1):
    print(f"  {i}. {doc['title']}")

✅ Created 6 banking policy documents

Documents:
  1. Commercial Loan Policy 2024
  2. Mortgage Lending Guidelines 2024
  3. Credit Card Policy 2024
  4. Business Banking Account Terms 2024
  5. Investment Services Guide 2024
  6. Risk Management and Fraud Prevention 2024


In [ ]:
financial_documents[0].keys()

dict_keys(['title', 'content'])

In [ ]:
print(financial_documents[0]["title"])
print(financial_documents[0]["content"])

Commercial Loan Policy 2024

        COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum 2 years
        - Annual revenue: Minimum $500,000

        Loan Amounts:
        - Minimum: $50,000
        - Maximum: $5,000,000
        - Term length: 5-25 years

        Interest Rates (as of November 2024):
        - Prime Rate + 2.5% to 5.5%
        - Current range: 8.5% - 11.5% APR
        - Fixed or variable rate options available

        Required Documentation:
        - Last 3 years business tax returns
        - Current year-to-date profit & loss statement
        - Business bank statements (6 months)
        - Business plan and financial projections
        - Personal financial statement from all owners with >20% stake

        Collateral Requirements:
        - Real estate: 75-80% loan-to-value ratio
        - Equipment: 70-75% loan-to-val

---

## 🔪 Step 3: Document Chunking

Split documents into smaller chunks for better retrieval.

In [ ]:
# Convert to LangChain Document objects
documents = [
    Document(
        page_content=doc['content'],
        metadata={
            'title': doc['title'],
            'source': doc['title'],
            'category': 'banking_policy'
        }
    )
    for doc in financial_documents
]

print(f"✅ Created {len(documents)} LangChain documents")

✅ Created 6 LangChain documents


In [ ]:
# documents  # this is a list of LAngChain Documents
documents[0]   # this is the 1st langchain document

Document(metadata={'title': 'Commercial Loan Policy 2024', 'source': 'Commercial Loan Policy 2024', 'category': 'banking_policy'}, page_content='\n        COMMERCIAL LOAN REQUIREMENTS\n\n        Minimum Qualifications:\n        - Personal credit score: 680 or higher\n        - Business credit score (Paydex): 75 or higher\n        - Time in business: Minimum 2 years\n        - Annual revenue: Minimum $500,000\n\n        Loan Amounts:\n        - Minimum: $50,000\n        - Maximum: $5,000,000\n        - Term length: 5-25 years\n\n        Interest Rates (as of November 2024):\n        - Prime Rate + 2.5% to 5.5%\n        - Current range: 8.5% - 11.5% APR\n        - Fixed or variable rate options available\n\n        Required Documentation:\n        - Last 3 years business tax returns\n        - Current year-to-date profit & loss statement\n        - Business bank statements (6 months)\n        - Business plan and financial projections\n        - Personal financial statement from all owner

In [ ]:
print(documents[0].page_content)


        COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum 2 years
        - Annual revenue: Minimum $500,000

        Loan Amounts:
        - Minimum: $50,000
        - Maximum: $5,000,000
        - Term length: 5-25 years

        Interest Rates (as of November 2024):
        - Prime Rate + 2.5% to 5.5%
        - Current range: 8.5% - 11.5% APR
        - Fixed or variable rate options available

        Required Documentation:
        - Last 3 years business tax returns
        - Current year-to-date profit & loss statement
        - Business bank statements (6 months)
        - Business plan and financial projections
        - Personal financial statement from all owners with >20% stake

        Collateral Requirements:
        - Real estate: 75-80% loan-to-value ratio
        - Equipment: 70-75% loan-to-value ratio
        - Inventory

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,          # Characters per chunk
    chunk_overlap=200,        # Overlap to preserve context
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""]
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print(f"\nExample chunk:")
print(f"Title: {chunks[0].metadata['title']}")
print(f"Content preview: {chunks[0].page_content[:200]}...")
print(f"Length: {len(chunks[0].page_content)} characters")

✅ Split into 17 chunks

Example chunk:
Title: Commercial Loan Policy 2024
Content preview: COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum ...
Length: 839 characters


In [ ]:
for chunk in chunks:
    print(f"Length: {len(chunk.page_content)} characters")

Length: 839 characters
Length: 529 characters
Length: 762 characters
Length: 788 characters
Length: 408 characters
Length: 774 characters
Length: 660 characters
Length: 545 characters
Length: 778 characters
Length: 738 characters
Length: 624 characters
Length: 802 characters
Length: 840 characters
Length: 629 characters
Length: 768 characters
Length: 673 characters
Length: 770 characters


---

## 🧠 Step 4: Create Embeddings

Initialize OpenAI embeddings model.

In [ ]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 1536 dimensions
    openai_api_key=openai_api_key
)

# Test embeddings with a sample query
test_text = "What are the requirements for a business loan?"
test_embedding = embeddings.embed_query(test_text)

print("✅ Embeddings model initialized")
print(f"\nTest embedding:")
print(f"  Query: '{test_text}'")
print(f"  Vector dimensions: {len(test_embedding)}")
print(f"  First 5 values: {test_embedding[:5]}")

✅ Embeddings model initialized

Test embedding:
  Query: 'What are the requirements for a business loan?'
  Vector dimensions: 1536
  First 5 values: [-0.044012051075696945, 0.04567737132310867, 0.05796181783080101, -0.010521800257265568, 0.020881393924355507]


---

## 🗄️ Step 5: Create Vector Stores

### 5.1 FAISS Vector Store (In-Memory)

In [ ]:
print("Creating FAISS vector store...")
start_time = time.time()

# Create FAISS vector store
faiss_vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

faiss_time = time.time() - start_time

print(f"✅ FAISS vector store created in {faiss_time:.2f} seconds")
print(f"   Total vectors: {len(chunks)}")

# Save FAISS index (optional - for reuse)
faiss_vectorstore.save_local("faiss_index")
print("✅ FAISS index saved to disk")

Creating FAISS vector store...
✅ FAISS vector store created in 1.44 seconds
   Total vectors: 17
✅ FAISS index saved to disk


### 5.2 ChromaDB Vector Store (Persistent)

In [ ]:
print("Creating ChromaDB vector store...")
start_time = time.time()

# Create ChromaDB vector store
chroma_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=r"./chroma_db",  # Persistent storage
    collection_name="banking_policies"
)

chroma_time = time.time() - start_time

print(f"✅ ChromaDB vector store created in {chroma_time:.2f} seconds")
print(f"   Total vectors: {len(chunks)}")
print(f"   Persisted to: ./chroma_db")

# Compare creation times
print(f"\n⏱️  Performance Comparison:")
print(f"   FAISS:    {faiss_time:.2f}s")
print(f"   ChromaDB: {chroma_time:.2f}s")

Creating ChromaDB vector store...
✅ ChromaDB vector store created in 1.25 seconds
   Total vectors: 17
   Persisted to: ./chroma_db

⏱️  Performance Comparison:
   FAISS:    1.44s
   ChromaDB: 1.25s


---

## 🔍 Step 6: Test Similarity Search

Compare FAISS and ChromaDB retrieval performance.

In [ ]:
# Test query
query = "What credit score is needed for a commercial loan?"

print(f"🔍 Query: '{query}'")
print("\n" + "="*80)

# FAISS search
print("\n📊 FAISS Results:")
print("="*80)
start_time = time.time()
faiss_results = faiss_vectorstore.similarity_search(query, k=3)
faiss_search_time = time.time() - start_time

for i, doc in enumerate(faiss_results, 1):
    print(f"\nResult {i}:")
    print(f"Source: {doc.metadata['title']}")
    print(f"Content: {doc.page_content[:200]}...")

print(f"\n⏱️  FAISS search time: {faiss_search_time*1000:.2f}ms")

🔍 Query: 'What credit score is needed for a commercial loan?'


📊 FAISS Results:

Result 1:
Source: Commercial Loan Policy 2024
Content: COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum ...

Result 2:
Source: Commercial Loan Policy 2024
Content: Collateral Requirements:
        - Real estate: 75-80% loan-to-value ratio
        - Equipment: 70-75% loan-to-value ratio
        - Inventory: 50-60% loan-to-value ratio
        - Personal guarantee ...

Result 3:
Source: Mortgage Lending Guidelines 2024
Content: RESIDENTIAL MORTGAGE LENDING POLICY

        Conventional Mortgages:
        - Minimum credit score: 620
        - Recommended credit score: 740+ for best rates
        - Down payment: 20% (or PMI req...

⏱️  FAISS search time: 339.92ms


In [ ]:
# ChromaDB search
print("\n" + "="*80)
print("\n📊 ChromaDB Results:")
print("="*80)
start_time = time.time()
chroma_results = chroma_vectorstore.similarity_search(query, k=3)
chroma_search_time = time.time() - start_time

for i, doc in enumerate(chroma_results, 1):
    print(f"\nResult {i}:")
    print(f"Source: {doc.metadata['title']}")
    print(f"Content: {doc.page_content[:200]}...")

print(f"\n⏱️  ChromaDB search time: {chroma_search_time*1000:.2f}ms")



📊 ChromaDB Results:

Result 1:
Source: Commercial Loan Policy 2024
Content: COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum ...

Result 2:
Source: Commercial Loan Policy 2024
Content: COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex): 75 or higher
        - Time in business: Minimum ...

Result 3:
Source: Commercial Loan Policy 2024
Content: Collateral Requirements:
        - Real estate: 75-80% loan-to-value ratio
        - Equipment: 70-75% loan-to-value ratio
        - Inventory: 50-60% loan-to-value ratio
        - Personal guarantee ...

⏱️  ChromaDB search time: 225.27ms


In [ ]:
# Performance comparison
print("\n" + "="*80)
print("\n⚡ Performance Comparison:")
print(f"   FAISS:    {faiss_search_time*1000:.2f}ms")
print(f"   ChromaDB: {chroma_search_time*1000:.2f}ms")
if faiss_search_time < chroma_search_time:
    speedup = chroma_search_time / faiss_search_time
    print(f"   FAISS is {speedup:.1f}x faster ⚡")
else:
    speedup = faiss_search_time / chroma_search_time
    print(f"   ChromaDB is {speedup:.1f}x faster ⚡")



⚡ Performance Comparison:
   FAISS:    339.92ms
   ChromaDB: 225.27ms
   ChromaDB is 1.5x faster ⚡


---

## 🤖 Step 7: Initialize LLM

Set up OpenAI's GPT-4o-mini for response generation.

In [ ]:
# Initialize ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.1,  # Low temperature for factual responses
    openai_api_key=openai_api_key
)

# Test LLM
test_response = llm.invoke("what is the Capital of India?")
print("✅ LLM initialized")
print(f"   Model: gpt-4.1-mini")
print(f"   Test response: {test_response.content}")

✅ LLM initialized
   Model: gpt-4.1-mini
   Test response: The capital of India is New Delhi.


---

## 🔗 Step 8: Create RAG Chain

### 8.1 Simple RAG Chain (FAISS)

In [ ]:
# Custom prompt template (LCEL style)
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a knowledgeable banking assistant. Use the retrieved context to "
                "answer user questions about banking policies, loans, and financial products.\n"
                "\nGuidelines:\n"
                "1. Use ONLY the provided context. If the answer is not in the context, say you don't know.\n"
                "2. Be precise and include numbers, rates, and requirements when available.\n"
                "3. Be professional and helpful.\n"
                "4. When possible, mention which policy document you used."
            ),
        ),
        (
            "human",
            "Question: {question}\n\nContext:\n{context}\n\nAnswer:",
        ),
    ]
)

def format_docs(docs: List[Document]) -> str:
    """Format retrieved documents into a string for the prompt."""
    formatted_chunks = []
    for doc in docs:
        title = doc.metadata.get("title", "Unknown Source")
        formatted_chunks.append(f"Source: {title}\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted_chunks)


# Create retrievers
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 3})
chroma_retriever = chroma_vectorstore.as_retriever(search_kwargs={"k": 3})

def make_rag_chain(retriever):
    """Create a simple RAG chain using LCEL (prompt | llm)."""
    return RunnableParallel(
        answer=(
            {
                "context": retriever | RunnableLambda(format_docs),
                "question": RunnablePassthrough(),
            }
            | qa_prompt
            | llm
            | StrOutputParser()
        ),
        # We'll also expose the raw source documents for inspection
        source_documents=retriever,
    )

# RAG chains for FAISS and ChromaDB
faiss_rag_chain = make_rag_chain(faiss_retriever)
chroma_rag_chain = make_rag_chain(chroma_retriever)

print("✅ FAISS & ChromaDB RAG chains created using LCEL (prompt | llm)")

✅ FAISS & ChromaDB RAG chains created using LCEL (prompt | llm)


### 8.2 Test FAISS RAG Chain

In [ ]:
# Test question
question = "What credit score is required for a commercial loan and what documentation is needed?"

print(f"\n{'='*80}")
print(f"💬 Question: {question}")
print(f"{'='*80}")

# Get answer from LCEL RAG chain (FAISS)
result = faiss_rag_chain.invoke(question)

print("\n🤖 Answer:")
print(result["answer"])

print(f"\n{'='*80}")
print("📚 Source Documents:")
print(f"{'='*80}")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n{i}. {doc.metadata.get('title', 'Unknown Source')}")
    print(f"   Excerpt: {doc.page_content[:150]}...")


💬 Question: What credit score is required for a commercial loan and what documentation is needed?

🤖 Answer:
For a commercial loan, the required credit scores are as follows (Commercial Loan Policy 2024):

- Personal credit score: 680 or higher
- Business credit score (Paydex): 75 or higher

Additionally, the business must have been operating for a minimum of 2 years with annual revenue of at least $500,000.

The required documentation includes:

- Last 3 years of business tax returns
- Current year-to-date profit & loss statement
- Business bank statements for the past 6 months
- Business plan and financial projections
- Personal financial statement from all owners holding more than 20% ownership

This information is based on the Commercial Loan Policy 2024.

📚 Source Documents:

1. Commercial Loan Policy 2024
   Excerpt: COMMERCIAL LOAN REQUIREMENTS

        Minimum Qualifications:
        - Personal credit score: 680 or higher
        - Business credit score (Paydex):...

2. Commer

In [ ]:
# Test question
question = "What did I ask you?"

print(f"\n{'='*80}")
print(f"💬 Question: {question}")
print(f"{'='*80}")

# Get answer from LCEL RAG chain (FAISS)
result = faiss_rag_chain.invoke(question)

print("\n🤖 Answer:")
print(result["answer"])

print(f"\n{'='*80}")
print("📚 Source Documents:")
print(f"{'='*80}")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n{i}. {doc.metadata.get('title', 'Unknown Source')}")
    print(f"   Excerpt: {doc.page_content[:150]}...")


💬 Question: What did I ask you?

🤖 Answer:
You asked me, "What did I ask you?"

📚 Source Documents:

1. Business Banking Account Terms 2024
   Excerpt: Merchant Services:
        - Credit card processing: 2.6% + $0.10 per transaction
        - ACH processing: $0.25 per transaction
        - Monthly eq...

2. Investment Services Guide 2024
   Excerpt: WEALTH MANAGEMENT AND INVESTMENT SERVICES

        Brokerage Accounts:
        - Account minimum: $0 for self-directed, $25,000 for managed
        - ...

3. Investment Services Guide 2024
   Excerpt: Financial Planning Services:
        - Comprehensive financial plan: $2,500 (one-time)
        - Retirement planning: $1,500 (one-time)
        - Esta...


---
## 9. Conversational RAG (Forward Reference)

The original notebook included a `ConversationalRetrievalChain` here, but that path uses the deprecated `ConversationBufferMemory` API. **Do not use it for new work.**

For multi-turn conversational RAG, wrap the chain you just built (Section 8) with the modern `RunnableWithMessageHistory` pattern from **Lab 3.1 § Memory**:

```python
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store: dict[str, InMemoryChatMessageHistory] = {}

def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

rag_with_history = RunnableWithMessageHistory(
    rag_chain, get_history,
    input_messages_key='question',
    history_messages_key='history',
)
rag_with_history.invoke(
    {'question': 'What's the prepayment fee?'},
    config={'configurable': {'session_id': 'user-42'}},
)
```

Add a `MessagesPlaceholder('history')` to the prompt and you have full multi-turn RAG with the modern API. The full pattern (with question contextualisation against history) is implemented in Lab 5.2 / Module 5.


---

## 📊 Step 9: Compare Vector Stores

Side-by-side comparison of FAISS vs ChromaDB.

In [ ]:
# Test questions
test_questions = [
    "What is the minimum down payment for an FHA loan?",
    "What are the fees for a business checking account?",
    "What interest rate do CDs offer?"
]

print(f"\n{'='*80}")
print("🏁 FAISS vs ChromaDB COMPARISON")
print(f"{'='*80}\n")

faiss_total_time = 0
chroma_total_time = 0

for i, question in enumerate(test_questions, 1):
    print(f"\nTest {i}: {question}")
    print("-" * 80)

    # FAISS
    start = time.time()
    faiss_result = faiss_rag_chain.invoke(question)
    faiss_time = time.time() - start
    faiss_total_time += faiss_time

    # ChromaDB
    start = time.time()
    chroma_result = chroma_rag_chain.invoke(question)
    chroma_time = time.time() - start
    chroma_total_time += chroma_time

    print(f"  FAISS:    {faiss_time*1000:6.2f}ms")
    print(f"  ChromaDB: {chroma_time*1000:6.2f}ms")
    print(f"  Winner:   {'FAISS ⚡' if faiss_time < chroma_time else 'ChromaDB ⚡'}")

print(f"\n{'='*80}")
print("📊 OVERALL RESULTS")
print(f"{'='*80}")
print(f"\nTotal Time (3 queries):")
print(f"  FAISS:    {faiss_total_time*1000:.2f}ms")
print(f"  ChromaDB: {chroma_total_time*1000:.2f}ms")
print(f"\nAverage Time per Query:")
print(f"  FAISS:    {faiss_total_time/len(test_questions)*1000:.2f}ms")
print(f"  ChromaDB: {chroma_total_time/len(test_questions)*1000:.2f}ms")

if faiss_total_time < chroma_total_time:
    speedup = chroma_total_time / faiss_total_time
    print(f"\n🏆 FAISS is {speedup:.2f}x faster overall!")
else:
    speedup = faiss_total_time / chroma_total_time
    print(f"\n🏆 ChromaDB is {speedup:.2f}x faster overall!")


🏁 FAISS vs ChromaDB COMPARISON


Test 1: What is the minimum down payment for an FHA loan?
--------------------------------------------------------------------------------
  FAISS:    2026.61ms
  ChromaDB: 1695.34ms
  Winner:   ChromaDB ⚡

Test 2: What are the fees for a business checking account?
--------------------------------------------------------------------------------
  FAISS:    2362.24ms
  ChromaDB: 2301.07ms
  Winner:   ChromaDB ⚡

Test 3: What interest rate do CDs offer?
--------------------------------------------------------------------------------
  FAISS:    3404.96ms
  ChromaDB: 4458.40ms
  Winner:   FAISS ⚡

📊 OVERALL RESULTS

Total Time (3 queries):
  FAISS:    7793.81ms
  ChromaDB: 8454.81ms

Average Time per Query:
  FAISS:    2597.94ms
  ChromaDB: 2818.27ms

🏆 FAISS is 1.08x faster overall!


---

## 🎯 Step 10: Interactive Q&A

Try your own questions!

---
## 10. NEW: Source Citations — Returning Page Numbers and Document IDs

A RAG answer that just gives you the text is unauditable — production users (especially in financial services) need to see *which document, which page* the answer came from. The fix: don't throw away the `Document` objects after retrieval. Build a chain that returns the answer **and** the source documents.

We'll do this with `RunnableParallel` so both branches run from the same input:
- One branch: build the prompt → LLM → answer string.
- Other branch: pass through the retrieved Documents (with their `metadata['source']`, `metadata['page']`) so the caller sees which docs were used.


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Same retriever you built earlier
retriever = faiss_vectorstore.as_retriever(search_kwargs={'k': 3})

# Prompt template: includes the docs we'll inject as context
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a banking-policy assistant. Answer the question using ONLY the context below. '
               'If the context does not contain the answer, say so. Be concise.'),
    ('user', 'Context:\n{context}\n\nQuestion: {question}'),
])

llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

def format_docs(docs):
    """Render retrieved docs as a numbered context block for the LLM."""
    return '\n\n'.join(
        f'[doc {i+1}] {d.page_content}'
        for i, d in enumerate(docs)
    )


In [ ]:
# --- Chain that returns BOTH the answer and the source docs ---
rag_chain_with_sources = RunnableParallel(
    {
        'docs': retriever,
        'question': RunnablePassthrough(),
    }
).assign(
    answer=(
        {
            'context': lambda x: format_docs(x['docs']),
            'question': lambda x: x['question'],
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
)

# Run it
result = rag_chain_with_sources.invoke('What is the prepayment fee for a personal loan?')

print('=== Answer ===')
print(result['answer'])
print()
print('=== Sources ===')
for i, doc in enumerate(result['docs'], 1):
    src = doc.metadata.get('source', 'unknown')
    page = doc.metadata.get('page', 'n/a')
    print(f'[doc {i}] source={src}  page={page}')
    print(f'        snippet: {doc.page_content[:120]}...')


In [ ]:
# --- Inline citations: ask the LLM to cite [doc N] markers in its answer ---
cited_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a banking-policy assistant. Answer using ONLY the context. '
               'After EVERY factual claim, cite the supporting doc(s) inline as [doc N]. '
               'If the context does not answer the question, say so explicitly.'),
    ('user', 'Context:\n{context}\n\nQuestion: {question}'),
])

cited_chain = RunnableParallel(
    {'docs': retriever, 'question': RunnablePassthrough()}
).assign(
    answer=(
        {'context': lambda x: format_docs(x['docs']), 'question': lambda x: x['question']}
        | cited_prompt
        | llm
        | StrOutputParser()
    )
)

result = cited_chain.invoke('What documentation is required for a personal loan?')
print('=== Cited Answer ===')
print(result['answer'])
print()
print('=== Source Map ===')
for i, doc in enumerate(result['docs'], 1):
    print(f'[doc {i}] {doc.metadata}')


### Why this matters in production

- **Auditability.** Compliance and legal teams require traceability — every claim → which doc, which page.
- **Debuggability.** When the answer is wrong, you immediately see which retrieved doc was at fault — bad chunk? wrong doc retrieved? hallucination over correct context? The source map tells you.
- **UI display.** Citations render as clickable links/footnotes in the chat UI, so the user can verify any claim themselves.
- **Hallucination signal.** Watch for answers that contain facts NOT covered by any retrieved doc — that's a grounding failure, often fixed by raising `k` or improving retrieval (Module 5).

**Production tip.** Always log `result['docs']` alongside `result['answer']` to your trace store. When users complain about wrong answers, you replay with the exact retrieved context.


In [ ]:
def ask_banking_question(question: str, use_faiss: bool = True):
    """Ask a question to the banking RAG system using LCEL chains.

    Args:
        question: Your question
        use_faiss: True for FAISS, False for ChromaDB
    """
    vectorstore_name = "FAISS" if use_faiss else "ChromaDB"
    rag_chain = faiss_rag_chain if use_faiss else chroma_rag_chain

    print(f"\n{'='*80}")
    print(f"🔍 Using {vectorstore_name} Vector Store")
    print(f"{'='*80}")
    print(f"\n💬 Question: {question}")
    print(f"\n{'='*80}\n")

    # Invoke the appropriate RAG chain
    result = rag_chain.invoke(question)

    print("🤖 Answer:\n")
    print(result["answer"])

    print(f"\n{'-'*80}")
    print("📚 Top Source Documents:")
    for i, doc in enumerate(result["source_documents"][:3], 1):
        print(f"\n{i}. {doc.metadata.get('title', 'Unknown Source')}")
        print(f"   {doc.page_content[:200]}...")

In [ ]:
# Try some questions!
sample_questions = [
    "What is the current mortgage rate for a 30-year fixed loan?",
    "What are the benefits of the Premium Rewards credit card?",
    "How much can I contribute to a Traditional IRA in 2024?"
]

# Test with FAISS
for q in sample_questions:
    ask_banking_question(q, use_faiss=False)
    time.sleep(1)


🔍 Using ChromaDB Vector Store

💬 Question: What is the current mortgage rate for a 30-year fixed loan?


🤖 Answer:

The current mortgage rate for a 30-year fixed loan is as follows, based on the type of mortgage:

- Conventional 30-year fixed: 6.5% APR
- FHA 30-year fixed: 6.25% APR
- VA 30-year fixed: 6.0% APR
- Jumbo 30-year fixed (loan amounts exceeding $726,200): 6.75% APR

These rates are from the Mortgage Lending Guidelines 2024, current as of November 2024.

--------------------------------------------------------------------------------
📚 Top Source Documents:

1. Mortgage Lending Guidelines 2024
   RESIDENTIAL MORTGAGE LENDING POLICY

        Conventional Mortgages:
        - Minimum credit score: 620
        - Recommended credit score: 740+ for best rates
        - Down payment: 20% (or PMI req...

2. Mortgage Lending Guidelines 2024
   RESIDENTIAL MORTGAGE LENDING POLICY

        Conventional Mortgages:
        - Minimum credit score: 620
        - Recommended credit score:

### Try Your Own Questions!

In [ ]:
# Ask your own question here!
my_question = "What documents do I need to apply for a mortgage?"  # Change this!

# Using FAISS
ask_banking_question(my_question, use_faiss=True)

# Uncomment to try with ChromaDB
# ask_banking_question(my_question, use_faiss=False)


🔍 Using FAISS Vector Store

💬 Question: What documents do I need to apply for a mortgage?


🤖 Answer:

To apply for a mortgage, you will need to provide the following documents as per the Mortgage Lending Guidelines 2024:

- Last 2 years W-2s or tax returns  
- Last 2 months pay stubs  
- Last 2 months bank statements  
- Employment verification  
- Photo ID  
- List of assets and liabilities  

These documents are required regardless of the mortgage type (Conventional, VA, Jumbo, FHA) to verify your income, employment, and financial status.

--------------------------------------------------------------------------------
📚 Top Source Documents:

1. Mortgage Lending Guidelines 2024
   VA Loans:
        - No minimum credit score (typically 620 recommended)
        - No down payment required
        - No mortgage insurance
        - Funding fee: 2.3% (can be financed)
        - Curre...

2. Mortgage Lending Guidelines 2024
   RESIDENTIAL MORTGAGE LENDING POLICY

        Conventional Mor

---

## 💾 Step 11: Save and Load Vector Stores

Persist your work for later use.

In [ ]:
# FAISS: Save
print("Saving FAISS index...")
faiss_vectorstore.save_local("banking_faiss_index")
print("✅ FAISS index saved to: banking_faiss_index/")

# FAISS: Load
print("\nLoading FAISS index...")
loaded_faiss = FAISS.load_local(
    "banking_faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print("✅ FAISS index loaded successfully")

# Test loaded index
test_results = loaded_faiss.similarity_search("credit score requirements", k=2)
print(f"\nTest search: Found {len(test_results)} results")
print(f"First result: {test_results[0].metadata['title']}")

print("\n" + "="*80)

# ChromaDB: Already persistent!
print("\nChromaDB info:")
print(f"✅ ChromaDB is already persistent at: ./chroma_db")
print(f"   Collection: banking_policies")
print(f"\nTo reload ChromaDB:")
print("""chroma_vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="banking_policies"
)""")

Saving FAISS index...
✅ FAISS index saved to: banking_faiss_index/

Loading FAISS index...
✅ FAISS index loaded successfully

Test search: Found 2 results
First result: Commercial Loan Policy 2024


ChromaDB info:
✅ ChromaDB is already persistent at: ./chroma_db
   Collection: banking_policies

To reload ChromaDB:
chroma_vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
    collection_name="banking_policies"
)


---

## 📈 Step 12: Performance Analysis

Analyze retrieval quality and speed.

In [ ]:
import numpy as np

def benchmark_vectorstore(vectorstore, queries: List[str], k: int = 3):
    """Benchmark vector store performance"""
    times = []

    for query in queries:
        start = time.time()
        results = vectorstore.similarity_search(query, k=k)
        elapsed = time.time() - start
        times.append(elapsed * 1000)  # Convert to ms

    return {
        'mean': np.mean(times),
        'median': np.median(times),
        'min': np.min(times),
        'max': np.max(times),
        'std': np.std(times)
    }

# Test queries
benchmark_queries = [
    "credit score requirements",
    "interest rates",
    "down payment",
    "business loan documentation",
    "credit card rewards",
    "investment account fees",
    "mortgage closing costs",
    "fraud protection",
    "retirement account limits",
    "business checking fees"
]

print("\n" + "="*80)
print("📊 VECTOR STORE PERFORMANCE BENCHMARK")
print("="*80)
print(f"\nBenchmarking with {len(benchmark_queries)} queries...\n")

# Benchmark FAISS
print("Testing FAISS...")
faiss_stats = benchmark_vectorstore(faiss_vectorstore, benchmark_queries)

# Benchmark ChromaDB
print("Testing ChromaDB...")
chroma_stats = benchmark_vectorstore(chroma_vectorstore, benchmark_queries)

# Display results
print("\n" + "="*80)
print("RESULTS (times in milliseconds)")
print("="*80)

print("\n📊 FAISS Statistics:")
print(f"   Mean:   {faiss_stats['mean']:.2f}ms")
print(f"   Median: {faiss_stats['median']:.2f}ms")
print(f"   Min:    {faiss_stats['min']:.2f}ms")
print(f"   Max:    {faiss_stats['max']:.2f}ms")
print(f"   Std:    {faiss_stats['std']:.2f}ms")

print("\n📊 ChromaDB Statistics:")
print(f"   Mean:   {chroma_stats['mean']:.2f}ms")
print(f"   Median: {chroma_stats['median']:.2f}ms")
print(f"   Min:    {chroma_stats['min']:.2f}ms")
print(f"   Max:    {chroma_stats['max']:.2f}ms")
print(f"   Std:    {chroma_stats['std']:.2f}ms")

# Winner
print("\n" + "="*80)
if faiss_stats['mean'] < chroma_stats['mean']:
    speedup = chroma_stats['mean'] / faiss_stats['mean']
    print(f"🏆 FAISS is {speedup:.2f}x faster on average!")
else:
    speedup = faiss_stats['mean'] / chroma_stats['mean']
    print(f"🏆 ChromaDB is {speedup:.2f}x faster on average!")
print("="*80)


📊 VECTOR STORE PERFORMANCE BENCHMARK

Benchmarking with 10 queries...

Testing FAISS...
Testing ChromaDB...

RESULTS (times in milliseconds)

📊 FAISS Statistics:
   Mean:   180.12ms
   Median: 172.45ms
   Min:    118.54ms
   Max:    342.51ms
   Std:    66.18ms

📊 ChromaDB Statistics:
   Mean:   232.79ms
   Median: 151.92ms
   Min:    111.87ms
   Max:    702.08ms
   Std:    190.31ms

🏆 FAISS is 1.29x faster on average!


---

## 🎓 Summary & Key Takeaways

### What We Built:
✅ Complete RAG pipeline for banking policy Q&A  
✅ Two vector stores: FAISS (fast) and ChromaDB (persistent)  
✅ OpenAI embeddings and GPT-4o-mini for generation  
✅ Conversational memory for multi-turn interactions  
✅ Performance benchmarking and comparison  

### Key Learnings:

**FAISS:**
- ⚡ Fastest retrieval (in-memory)
- 💰 Free and open-source
- 🔧 Requires manual persistence
- ✅ Best for: Prototypes, development, single-machine

**ChromaDB:**
- 💾 Automatic persistence
- 📊 Built-in metadata filtering
- 🔍 Good for production with persistence needs
- ✅ Best for: Applications needing data persistence

**OpenAI Models:**
- text-embedding-3-small: Excellent quality, affordable
- gpt-4o-mini: Fast, cost-effective, high quality

### When to Use What:

**Use FAISS when:**
- Prototyping and testing
- Speed is critical
- Dataset fits in memory
- Single-machine deployment

**Use ChromaDB when:**
- Need persistent storage
- Building production applications
- Want easy setup
- Need metadata filtering

### Next Steps:
1. Try the second notebook: OpenSearch + Bedrock for production RAG
2. Add your own financial documents
3. Experiment with different chunk sizes
4. Implement hybrid search (semantic + keyword)
5. Add evaluation metrics for answer quality

---

## 🚀 Great job completing Hands-On #1!

You now have a working RAG system that can answer banking questions using financial documents.

**Ready for production-scale RAG?** → Move to Hands-On #2: OpenSearch + Bedrock! 💪

---
## 12. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **End-to-end RAG = retriever + prompt + LLM + parser** | LCEL composes them into one runnable; same shape regardless of vector store |
| **FAISS vs ChromaDB are interchangeable behind `as_retriever()`** | Picking one is a deployment decision, not a code-rewrite |
| **Chunk size + retrieval k drive answer quality** | 800 tokens / 100 overlap / k=3 is a reasonable starting point — tune empirically |
| **Source citations make answers auditable** | Use `RunnableParallel` to return both the answer and the supporting Documents |
| **Persist embeddings** | FAISS `save_local()` / ChromaDB persist_directory — never re-embed cold start in production |
| **Conversational RAG = chain + RunnableWithMessageHistory** | Modern memory pattern from Lab 3.1; do not use deprecated `ConversationalRetrievalChain` |

**Next Lab:** Lab 4.4 — Amazon Bedrock Knowledge Bases (Managed RAG) ☁️


## 13. Stretch Exercise (Optional)

1. Wrap your `rag_chain_with_sources` in `RunnableWithMessageHistory` so the bot remembers the user's previous turns. Include a system instruction to expand follow-up questions using the chat history.
2. Add a `where=` metadata filter to the ChromaDB retriever — restrict retrieval to one product (e.g., `personal_loan`) — and verify queries about other products return 'no information'.
3. Build a 10-question eval set (banking-policy QA pairs). Run the FAISS vs ChromaDB chains side-by-side; report wins/ties/losses + average answer latency.
4. Add a *retrieval-quality circuit breaker*: if the top-1 retrieved doc has cosine similarity < 0.6, return 'I don't have that information' instead of letting the LLM guess.
5. Replace `OpenAIEmbeddings` with `BedrockEmbeddings(model_id='amazon.titan-embed-text-v2:0')` and rerun the full pipeline. Compare retrieval quality on 5 banking queries.
6. Stream the answer with `.stream()` while accumulating sources separately — preview of an SSE chat UI (Module 10).
